In [1]:
from pathlib import Path
import re

data_root = Path(r"J:\project_trainingAggression\Data")
updated_root = Path(r"A:\processedData")

DRY_RUN = False

date_re = re.compile(r"\d{8}")
day_re = re.compile(r"Day\d+", re.IGNORECASE)
obs_re = re.compile(r"obs\d+", re.IGNORECASE)
mouse_re = re.compile(r"mouse(\d+)|m(\d+)", re.IGNORECASE)

date_cache = {}
missing_maps = set()
collisions = []


def get_mouse_number(text):
    m = mouse_re.search(text)
    return (m.group(1) or m.group(2)) if m else None


def get_obs(text):
    m = obs_re.search(text)
    return m.group(0).lower() if m else None


def get_day_index_and_name(rel_path):
    for i, part in enumerate(rel_path.parts):
        if day_re.fullmatch(part):
            return i, part
    return None, None


def find_mouse_day_dirs(root, mouse_number, day_name):
    out = []
    for p in root.rglob("*"):
        if p.is_dir() and mouse_number in p.name:
            d = p / day_name
            if d.is_dir():
                out.append(d)
    return out


def find_single_updated_date(mouse_number, day_name, obs_id):
    dates = set()

    for day_dir in find_mouse_day_dirs(updated_root, mouse_number, day_name):
        for p in day_dir.rglob("*"):
            rel_to_day = str(p.relative_to(day_dir))
            if obs_id.lower() in rel_to_day.lower():
                dates.update(date_re.findall(rel_to_day))

    dates = sorted(dates)
    return dates[0] if len(dates) == 1 else None


def new_name_for(path):
    rel = path.relative_to(data_root)
    rel_str = str(rel)

    day_idx, day_name = get_day_index_and_name(rel)
    if day_idx is None:
        return None

    # Only rename things AFTER DayXX
    if len(rel.parts) <= day_idx + 1:
        return None

    if not date_re.search(path.name):
        return None

    mouse_number = get_mouse_number(rel_str)
    obs_id = get_obs(rel_str)

    if mouse_number is None or obs_id is None:
        return None

    key = (mouse_number, day_name, obs_id)

    if key not in date_cache:
        date_cache[key] = find_single_updated_date(mouse_number, day_name, obs_id)

    new_date = date_cache[key]

    if new_date is None:
        missing_maps.add(key)
        return None

    new_name = date_re.sub(new_date, path.name)
    return new_name if new_name != path.name else None


def try_rename(path, kind):
    new_name = new_name_for(path)
    if new_name is None:
        return 0

    new_path = path.with_name(new_name)

    print(f"RENAME {kind}:")
    print(f"  FROM: {path}")
    print(f"  TO:   {new_path}")
    print()

    if DRY_RUN:
        return 1

    if not path.exists():
        return 0

    if new_path.exists():
        collisions.append((path, new_path))
        print(f"SKIP collision, target exists: {new_path}")
        print()
        return 0

    path.rename(new_path)
    return 1


# -------------------------
# 1. Rename folders first, shallow to deep.
# Repeat until no more folder renames happen.
# -------------------------
total_dir_renames = 0

while True:
    dirs = [p for p in data_root.rglob("*") if p.is_dir()]
    dirs.sort(key=lambda p: len(p.parts))

    n = 0
    for d in dirs:
        n += try_rename(d, "FOLDER")

    total_dir_renames += n

    if n == 0 or DRY_RUN:
        break


# -------------------------
# 2. Rename files after folders
# -------------------------
files = [p for p in data_root.rglob("*") if p.is_file()]
files.sort(key=lambda p: len(p.parts), reverse=True)

total_file_renames = 0
for f in files:
    total_file_renames += try_rename(f, "FILE")


print("Done.")
print(f"DRY_RUN = {DRY_RUN}")
print(f"Folder renames: {total_dir_renames}")
print(f"File renames: {total_file_renames}")
print(f"Missing mappings: {len(missing_maps)}")
print(f"Collisions skipped: {len(collisions)}")

for old, new in collisions[:50]:
    print("COLLISION:")
    print(f"  OLD: {old}")
    print(f"  NEW: {new}")

RENAME FOLDER:
  FROM: J:\project_trainingAggression\Data\20250817_mouse975826\Day19\neuralData\catgt_20250818_m975826_obs6_g0
  TO:   J:\project_trainingAggression\Data\20250817_mouse975826\Day19\neuralData\catgt_20250911_m975826_obs6_g0

RENAME FOLDER:
  FROM: J:\project_trainingAggression\Data\20250817_mouse975827\Day01\neuralData\catgt_20250817_m975827_obs1_g0
  TO:   J:\project_trainingAggression\Data\20250817_mouse975827\Day01\neuralData\catgt_20250824_m975827_obs1_g0

RENAME FOLDER:
  FROM: J:\project_trainingAggression\Data\20250817_mouse975827\Day03\neuralData\catgt_20250818_m975827_obs2_g0
  TO:   J:\project_trainingAggression\Data\20250817_mouse975827\Day03\neuralData\catgt_20250826_m975827_obs2_g0

RENAME FOLDER:
  FROM: J:\project_trainingAggression\Data\20250817_mouse975827\Day05\neuralData\catgt_20250818_m975827_obs3_g0
  TO:   J:\project_trainingAggression\Data\20250817_mouse975827\Day05\neuralData\catgt_20250828_m975827_obs3_g0

RENAME FOLDER:
  FROM: J:\project_traini